In [1]:
import polars as pl
import polars.selectors as cs
import plotly.graph_objects as go
import plotly.express as px
from typing import Union, Optional

from polars_reverse_geocode import find_closest_city, find_closest_state

In [2]:
df = pl.read_parquet('../data/raw/tripdata.parquet')

In [3]:
df.shape

(6171785, 13)

In [4]:
df.schema

Schema([('ride_id', String),
        ('rideable_type', String),
        ('started_at', String),
        ('ended_at', String),
        ('start_station_name', String),
        ('start_station_id', String),
        ('end_station_name', String),
        ('end_station_id', String),
        ('start_lat', Float64),
        ('start_lng', Float64),
        ('end_lat', Float64),
        ('end_lng', Float64),
        ('member_casual', String)])

In [5]:
df.describe()

statistic,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
str,str,str,str,str,str,str,str,str,f64,f64,f64,f64,str
"""count""","""6171785""","""6171785""","""6171785""","""6171785""","""4882299""","""4882299""","""4816298""","""4816298""",6.171785e6,6.171785e6,6.166334e6,6.166334e6,"""6171785"""
"""null_count""","""0""","""0""","""0""","""0""","""1289486""","""1289486""","""1355487""","""1355487""",0.0,0.0,5451.0,5451.0,"""0"""
"""mean""",null,null,null,null,null,null,null,null,41.904583,-87.646934,41.904974,-87.647292,null
"""std""",null,null,null,null,null,null,null,null,0.044268,0.026953,0.044441,0.027212,null
"""min""","""00000052CAB9923E""","""classic_bike""","""2025-09-30 08:37:53.455""","""2025-10-01 00:00:07.199""","""11th St Ped Bridge""",""" CHI02331""","""2112 W Peterson Ave""",""" CHI02331""",41.64,-87.91,41.51,-88.1,"""casual"""
"""25%""",null,null,null,null,null,null,null,null,41.882752,-87.66,41.882906,-87.66,null
"""50%""",null,null,null,null,null,null,null,null,41.9,-87.642884,41.9,-87.643118,null
"""75%""",null,null,null,null,null,null,null,null,41.93,-87.63,41.93,-87.63,null
"""max""","""FFFFF83BFBE37975""","""electric_bike""","""2026-09-30 23:59:38.090""","""2026-09-30 23:59:48.043""","""[Archived] Wilton Ave & Divers…","""CHI02422""","""[Archived] Wilton Ave & Divers…","""CHI02422""",42.07,-87.52,42.19,-87.37,"""member"""


In [6]:
NULL_TOKENS = ["", "NA", "N/A", "null", "-"]

df = df.with_columns(
    pl.when(pl.col(pl.String).str.strip_chars().is_in(NULL_TOKENS))
      .then(None)
      .otherwise(pl.col(pl.String))
      .name.keep(),
    cs.float().fill_nan(None),
)

In [18]:
def profile(d: pl.DataFrame):
    nulls = d.null_count().row(0)

    return pl.DataFrame({
        "column": df.columns,
        "dtype": [str(t) for t in df.dtypes],
        "nulls": nulls,
        "pct_null": [round(n / df.height * 100, 1) for n in nulls],
        "uniques": [df[c].n_unique() for c in df.columns],
    })

profile(df)

column,dtype,nulls,pct_null,uniques
str,str,i64,f64,i64
"""ride_id""","""String""",0,0.0,6171750
"""rideable_type""","""String""",0,0.0,2
"""started_at""","""String""",0,0.0,6170353
"""ended_at""","""String""",0,0.0,6168508
"""start_station_name""","""String""",1289486,20.9,1990
…,…,…,…,…
"""start_lat""","""Float64""",0,0.0,97278
"""start_lng""","""Float64""",0,0.0,95844
"""end_lat""","""Float64""",5451,0.1,2184


In [53]:
df.filter(df.is_duplicated()).height

70

In [9]:
NULL_TOKENS = ["", "NA", "N/A", "null", "-"]

df = df.with_columns(
    pl.when(pl.col(pl.String).str.strip_chars().is_in(NULL_TOKENS))
      .then(None)
      .otherwise(pl.col(pl.String))
      .name.keep(),
    cs.float().fill_nan(None),
)

In [10]:
def null_per_group(d: pl.DataFrame, group: str):
    return d.group_by(group).agg(
        pl.all().is_null().mean().mul(100).round(1).name.suffix("_pct_nulls"),
        pl.len().alias("n"),
    )

In [29]:
df = df.with_columns(
    pl.concat_str(
        pl.col("started_at").str.to_datetime().dt.year(),
        pl.col("started_at").str.to_datetime().dt.month(),
        separator="."
    ).alias("year_month"),
)

nulls_per_year_month = null_per_group(df, "year_month")
nulls_per_year_month

year_month,ride_id_pct_nulls,rideable_type_pct_nulls,started_at_pct_nulls,ended_at_pct_nulls,start_station_name_pct_nulls,start_station_id_pct_nulls,end_station_name_pct_nulls,end_station_id_pct_nulls,start_lat_pct_nulls,start_lng_pct_nulls,end_lat_pct_nulls,end_lng_pct_nulls,member_casual_pct_nulls,year_pct_nulls,month_pct_nulls,n
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""2026.3""",0.0,0.0,0.0,0.0,18.2,18.2,19.8,19.8,0.0,0.0,0.1,0.1,0.0,0.0,0.0,317015
"""2026.2""",0.0,0.0,0.0,0.0,15.9,15.9,17.3,17.3,0.0,0.0,0.1,0.1,0.0,0.0,0.0,201454
"""2025.10""",0.0,0.0,0.0,0.0,22.4,22.4,23.8,23.8,0.0,0.0,0.1,0.1,0.0,0.0,0.0,646096
"""2025.9""",0.0,0.0,0.0,0.0,15.5,15.5,40.8,40.8,0.0,0.0,18.3,18.3,0.0,0.0,0.0,71
"""2026.1""",0.0,0.0,0.0,0.0,17.8,17.8,20.0,20.0,0.0,0.0,0.1,0.1,0.0,0.0,0.0,137782
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2026.7""",0.0,0.0,0.0,0.0,21.8,21.8,22.6,22.6,0.0,0.0,0.1,0.1,0.0,0.0,0.0,869101
"""2026.8""",0.0,0.0,0.0,0.0,21.8,21.8,22.8,22.8,0.0,0.0,0.1,0.1,0.0,0.0,0.0,868069
"""2026.9""",0.0,0.0,0.0,0.0,20.8,20.8,22.1,22.1,0.0,0.0,0.1,0.1,0.0,0.0,0.0,770511


In [12]:
nulls_per_rideable_type = null_per_group(df, "rideable_type")
nulls_per_rideable_type

rideable_type,ride_id_pct_nulls,started_at_pct_nulls,ended_at_pct_nulls,start_station_name_pct_nulls,start_station_id_pct_nulls,end_station_name_pct_nulls,end_station_id_pct_nulls,start_lat_pct_nulls,start_lng_pct_nulls,end_lat_pct_nulls,end_lng_pct_nulls,member_casual_pct_nulls,year_month_pct_nulls,n
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""electric_bike""",0.0,0.0,0.0,29.4,29.4,30.8,30.8,0.0,0.0,0.0,0.0,0.0,0.0,4387353
"""classic_bike""",0.0,0.0,0.0,0.0,0.0,0.3,0.3,0.0,0.0,0.3,0.3,0.0,0.0,1784432


In [13]:
nulls_per_user_type = null_per_group(df, "member_casual")
nulls_per_user_type

member_casual,ride_id_pct_nulls,rideable_type_pct_nulls,started_at_pct_nulls,ended_at_pct_nulls,start_station_name_pct_nulls,start_station_id_pct_nulls,end_station_name_pct_nulls,end_station_id_pct_nulls,start_lat_pct_nulls,start_lng_pct_nulls,end_lat_pct_nulls,end_lng_pct_nulls,year_month_pct_nulls,n
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""casual""",0.0,0.0,0.0,0.0,21.8,21.8,25.0,25.0,0.0,0.0,0.2,0.2,0.0,2189024
"""member""",0.0,0.0,0.0,0.0,20.4,20.4,20.3,20.3,0.0,0.0,0.0,0.0,0.0,3982761


In [52]:
members_pct = round(100 * df.filter(pl.col("member_casual") == "member").height / df.height, 2)

print(f"{members_pct}% of records belong to member users, and {100 - members_pct}% of records belong to casual users.")

64.53% of records belong to member users, and 35.47% of records belong to casual users.


In [14]:
df_electic_bikes = df.filter(
    pl.col("rideable_type") == "electric_bike"
)

nulls_per_user_type_electric = null_per_group(df_electic_bikes, "member_casual")
nulls_per_user_type_electric

member_casual,ride_id_pct_nulls,rideable_type_pct_nulls,started_at_pct_nulls,ended_at_pct_nulls,start_station_name_pct_nulls,start_station_id_pct_nulls,end_station_name_pct_nulls,end_station_id_pct_nulls,start_lat_pct_nulls,start_lng_pct_nulls,end_lat_pct_nulls,end_lng_pct_nulls,year_month_pct_nulls,n
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""casual""",0.0,0.0,0.0,0.0,29.1,29.1,33.2,33.2,0.0,0.0,0.0,0.0,0.0,1640338
"""member""",0.0,0.0,0.0,0.0,29.6,29.6,29.3,29.3,0.0,0.0,0.0,0.0,0.0,2747015


In [40]:
def plot_null_percentages_bar(
    d: pl.DataFrame,
    category_col: str,
    value_cols: Union[str, list[str]],
    title: str = "Missing Values Analysis",
    subtitle: Optional[str] = None,
    category_title: Optional[str] = None,
    value_title: str = "Percentage of Nulls (%)",
    color_title: Optional[str] = "Attribute",
    show_legend: Optional[bool] = None,
    barmode: str = "group",
    label_angle: int = 0,
    y_domain: list[float] = [0, 100],
    width: int = 800,
    height: int = 600
):
    if isinstance(value_cols, list):
        df_plot = d.unpivot(
            on=value_cols,
            index=category_col,
            variable_name="attribute",
            value_name="null_percentage"
        )
        y_col = "null_percentage"
        color_col = "attribute"

        if show_legend is None:
            show_legend = True
    else:
        df_plot = d.clone()
        y_col = value_cols
        color_col = category_col
        if show_legend is None:
            show_legend = False

    if category_col == "year_month":
        df_plot = (
            df_plot
            .with_columns(
                pl.col(category_col).str.split_exact(".", 1)
                .struct.rename_fields(["year", "month"])
                .alias("_date_parts")
            )
            .with_columns(
                pl.date(
                    pl.col("_date_parts").struct.field("year").cast(pl.Int32),
                    pl.col("_date_parts").struct.field("month").cast(pl.Int32),
                    1
                ).alias("_date")
            )
            .sort("_date")
            .with_columns(
                pl.col("_date").dt.strftime("%b %Y").alias(category_col)
            )
            .drop(["_date_parts", "_date"])
        )

    fig = px.bar(
        df_plot,
        x=category_col,
        y=y_col,
        color=color_col,
        barmode=barmode,
        width=width,
        height=height,
        color_discrete_sequence=["#4C78A8", "#F58518"],
        category_orders={
            category_col: df_plot[category_col].unique(maintain_order=True).to_list()
        },
        labels={
            category_col: category_title or category_col,
            y_col: value_title,
            color_col: color_title or ""
        }
    )

    title_text = f"<b>{title}</b>"

    if subtitle:
        title_text += (
            f"<br><span style='font-size:15px; "
            f"color:#333333; font-weight:normal;'>{subtitle}</span>"
        )

    fig.update_layout(
        template="plotly_white",

        font=dict(
            family="Arial, sans-serif",
            size=14,
            color="#111111"
        ),

        title=dict(
            text=title_text,
            x=0.02,
            xanchor="left",
            y=0.97,
            yanchor="top",
            font=dict(size=22, color="#111111")
        ),

        margin=dict(
            l=85,
            r=190,
            t=115,
            b=95
        ),

        xaxis=dict(
            title=dict(
                text=category_title or category_col,
                font=dict(size=16, color="#111111"),
                standoff=15
            ),
            tickangle=label_angle,
            tickfont=dict(size=12, color="#111111"),
            showgrid=False,
            showline=True,
            linecolor="#888888",
            ticks="outside",
            tickcolor="#888888",
            type="category",
            automargin=True
        ),

        yaxis=dict(
            title=dict(
                text=value_title,
                font=dict(size=16, color="#111111"),
                standoff=15
            ),
            range=y_domain,
            dtick=20,
            tickfont=dict(size=13, color="#111111"),
            showgrid=True,
            gridcolor="#EAEAEA",
            zeroline=False,
            showline=True,
            linecolor="#888888",
            automargin=True
        ),

        legend=dict(
            title=dict(
                text=color_title or "",
                font=dict(size=14, color="#111111")
            ),
            font=dict(size=13, color="#111111"),
            x=1.02,
            y=1,
            xanchor="left",
            yanchor="top",
            bgcolor="rgba(0,0,0,0)"
        ),

        showlegend=show_legend,
        plot_bgcolor="white",
        paper_bgcolor="white",
        bargap=0.12,
        bargroupgap=0.02
    )

    return fig

In [42]:
fig = plot_null_percentages_bar(
    d=nulls_per_year_month,
    category_col="year_month",
    value_cols=[
        "start_station_id_pct_nulls",
        "end_station_id_pct_nulls"
    ],
    category_title="Month",
    title="Missing Station Identifiers by Month",
    subtitle="Monthly comparison of null percentage for start_station_id and end_station_id",
    width=1100,
    height=700,
    label_angle=-45
)

fig.show()

In [44]:
fig1 = plot_null_percentages_bar(
    d=nulls_per_rideable_type,
    category_col='rideable_type',
    value_cols=[
        "start_station_id_pct_nulls",
        "end_station_id_pct_nulls"
    ],
    category_title='Rideable Type',
    title='Missing Station Identifiers by Rideable Type',
    subtitle='Comparison of null percentage for start_station_id and end_station_id by types of bicycle',
)
fig1.show()

In [43]:
fig1 = plot_null_percentages_bar(
    d=nulls_per_user_type_electric,
    category_col='member_casual',
    value_cols=[
        "start_station_id_pct_nulls",
        "end_station_id_pct_nulls"
    ],
    category_title='User Type',
    title='Missing End Station Identifiers for Electric Bikes by User Type',
    subtitle='Comparison of null percentage across member types',
)
fig1.show()

In [20]:
rides_less_than_1_min = round(100 * df.filter(
    pl.col("ended_at").str.to_datetime()
    .sub(pl.col("started_at").str.to_datetime())
    .dt.total_seconds() < 60
).height / df.height, 2)

print(f"{rides_less_than_1_min}% of rides are less than 1 minute long.")

2.58% of rides are less than 1 minute long.


In [12]:
clean_df = df.filter(
    pl.col("start_lat").is_not_null()
    & pl.col("start_lng").is_not_null()
    & pl.col("end_lat").is_not_null()
    & pl.col("end_lng").is_not_null()
)

route_summary = (
    clean_df.group_by(
        [
            "start_station_name",
            "end_station_name",
            "start_lat",
            "start_lng",
            "end_lat",
            "end_lng",
        ]
    )
    .agg(pl.len().alias("trip_count"))
    .sort("trip_count", descending=True)
    .head(500)
)

route_lats = []
route_longs = []
for row in route_summary.select(["start_lat", "start_lng", "end_lat", "end_lng"]).iter_rows():
    route_lats.extend([row[0], row[2], None])
    route_longs.extend([row[1], row[3], None])

In [64]:
fig_agg = go.Figure(
    go.Scattermap(
        lat=route_lats,
        lon=route_longs,
        mode="lines",
        line=dict(width=2, color="orange"),
        name="Top Routes",
    )
)

fig_agg.update_layout(
    map=dict(
        style="carto-positron",
        center=dict(lat=route_summary["start_lat"].mean(), lon=route_summary["start_lng"].mean()),
        zoom=11,
    ),
    margin=dict(l=0, r=0, t=0, b=0),
)

fig_agg.show()

In [15]:
# Considers a rectangular boundary around Chicago
lat_min, lat_max = 41.64, 42.03
long_min, long_max = -87.94, -87.52

clean_df = clean_df.with_columns(
    (pl.col("start_lat").is_between(lat_min, lat_max)
    & pl.col("start_lng").is_between(long_min, long_max))
    .alias("start_inside"),

    (pl.col("end_lat").is_between(lat_min, lat_max)
    & pl.col("end_lng").is_between(long_min, long_max))
    .alias("end_inside"),
)

clean_df.select(
    (pl.col("start_inside") & pl.col("end_inside")).sum().alias("inside"),
    (pl.col("start_inside") & pl.col("end_inside")).mean().mul(100).round(2).alias("inside_pct"),
    (~pl.col("start_inside") & ~pl.col("end_inside")).sum().alias("outside"),
    (~pl.col("start_inside") & ~pl.col("end_inside")).mean().mul(100).round(2).alias("outside_pct"),
    (pl.col("start_inside") != pl.col("end_inside")).sum().alias("mixed"),
    (pl.col("start_inside") != pl.col("end_inside")).mean().mul(100).round(2).alias("mixed_pct"),
)

inside,inside_pct,outside,outside_pct,mixed,mixed_pct
u32,f64,u32,f64,u32,f64
6099189,98.91,52052,0.84,15093,0.24
